# Nudged Elastic Band (NEB)

Calculate the reaction energy barrier and energy profile along a reaction path using Quantum ESPRESSO `neb.x` on the Mat3ra platform.

## Prerequisites

Before running this notebook, put the NEB path structures on the platform in an **ordered materials set**:

1. **First** member = initial image (required).
2. **Last** member = final image (required).
3. **Middle** members = intermediate images (optional).
4. If the set has only first+last, set `N_IMAGES` (e.g. `20`) so Quantum ESPRESSO interpolates intermediates.

Path order follows the ordered-set indices (same as the job designer), not material names.

<h2 style="color:green">Usage</h2>

1. Set `MATERIAL_SET` in cell 1.2 and NEB-specific settings (`N_IMAGES`, k-grid) in cell 1.3 below.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the reaction energy profile.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure materials lookup, workflow, compute, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Resolve NEB path materials from an ordered materials set (first, optional intermediates, last).
1. Create workflow and set its parameters: load the NEB workflow from Standata, set k-grid / `N_IMAGES`, and save the workflow.
1. Configure compute: get list of clusters and create compute configuration.
1. Create the multi-material NEB job.
1. Submit the job and monitor the status.
1. Retrieve results: visualize the reaction energy profile along the path.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflow and job

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
# Set organization name to use it as the owner, otherwise your personal account is used
ORGANIZATION_NAME = None

# 3. Material parameters (prerequisite: ordered materials set on the platform)
# Order in the set: first = initial, middle = intermediates (optional), last = final
MATERIAL_SET = "H2+H"

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "neb.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Nudged Elastic Band (NEB)"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set specific NEB parameters

In [ ]:
# Intermediate count for QE when the set has only first+last (ignored if middle images exist)
N_IMAGES = None  # e.g. 20; defaults to 1 when exactly two materials are in the set

# K-grid for the NEB unit
NEB_KGRID = [1, 1, 1]

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate


await authenticate()

### 2.2. Initialize API Client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Resolve NEB path materials
### 3.1. Load materials from an ordered set
Members are loaded in ordered-set index order: first → intermediates (if any) → last.

In [ ]:
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import list_materials_by_set
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

material_dicts = list_materials_by_set(client, ACCOUNT_ID, MATERIAL_SET)
if len(material_dicts) < 2:
    raise ValueError(
        f"Ordered set '{MATERIAL_SET}' must contain at least first and last images "
        f"(found {len(material_dicts)})."
    )
print(f"✅ Loaded {len(material_dicts)} material(s) from ordered set '{MATERIAL_SET}'")
materials = [Material.create(material_dict) for material_dict in material_dicts]
print(f"  first (initial): {materials[0].name} ({materials[0].id})")
if len(materials) > 2:
    for material in materials[1:-1]:
        print(f"  intermediate: {material.name} ({material.id})")
print(f"  last (final): {materials[-1].name} ({materials[-1].id})")
visualize(materials)

### 3.2. Use resolved materials for the job
Path materials already exist on the platform in the ordered set.

In [ ]:
saved_materials = materials
for saved_material in saved_materials:
    print(f"✅ Material: {saved_material.name} ({saved_material.id})")

## 4. Create workflow and set its parameters
### 4.1. Get list of applications and select one

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Create workflow from standard workflows and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Modify important settings
Set k-grid for the NEB unit. When the set has only first+last, set `N_IMAGES` (e.g. 20) so QE interpolates intermediates.

In [ ]:
from mat3ra.wode.context.providers import PointsGridDataProvider

neb_subworkflow = workflow.subworkflows[0]
unit_to_modify = neb_subworkflow.get_unit_by_name(name="neb")

if NEB_KGRID is not None:
    new_context_kgrid = PointsGridDataProvider(dimensions=NEB_KGRID, isEdited=True).get_context_item_data()
    unit_to_modify.add_context(new_context_kgrid)

effective_n_images = N_IMAGES
if len(saved_materials) == 2 and effective_n_images is None:
    effective_n_images = 1
if effective_n_images is not None:
    unit_to_modify.add_context(
        {"name": "neb", "isEdited": True, "data": {"nImages": effective_n_images}, "extraData": {}}
    )
    print(f"Using N_IMAGES={effective_n_images}")

neb_subworkflow.set_unit(unit_to_modify)
visualize_workflow(workflow)

### 4.4. Save workflow to collection

In [ ]:
from mat3ra.utils.namespace import dict_to_namespace_recursive
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration for the job


In [ ]:
from mat3ra.ide.compute import Compute

# Select cluster: use specified name if provided, otherwise use first available
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job with materials and workflow configuration
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.ui import display_JSON

print(f"Materials: {[m.id for m in saved_materials]}")
print(f"Workflow: {saved_workflow.id}")
print(f"Project: {project_id}")

path_label = MATERIAL_SET or "neb-path"
job_name = f"{MY_WORKFLOW_NAME} {path_label} {timestamp}"
job_response = create_job(
    api_client=client,
    materials=saved_materials,
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict()
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve results

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

profile_data = client.properties.get_for_job(job_id, property_name="reaction_energy_profile")
visualize_properties(profile_data, title="Reaction Energy Profile")

energies = profile_data[0]["yDataSeries"][0]
print(f"Peak reaction energy = {max(energies):.3f} eV")

In [ ]:
barrier_data = client.properties.get_for_job(job_id, property_name="reaction_energy_barrier")
visualize_properties(barrier_data, title="Reaction Energy Barrier")